# 2.4 Lab - Propagating Changes with CDF
Replace every `<FILL_IN>` and run the **CHECK** cell after each step. The solution is in `02-data-privacy/solutions/`.

**Scenario.** Customer master data lives in `customers_bronze`. Two downstream tables depend on it:
- `customers_silver`: analytics-ready, with the email reduced to its domain
- `clicks_silver`: click events keyed by `user_id`

The CRM sends daily CDC files with updates and deletes. You must keep both downstream tables in sync **incrementally** with Change Data Feed, and make deletes physical.

![Lab target: CRM CDC files are merged into customers_bronze (CDF on); a change feed stream with a checkpoint runs upsert_batch into customers_silver and clicks_silver; step 6 purges and vacuums](../images/2.4-lab-target-flow.svg)

**How to read it:** the step numbers in the boxes match the steps below. The dashed line on top is the one-off snapshot that seeds `customers_silver` before the stream starts, which is why the stream itself must skip version 1. Step 5 reruns the same stream, which resumes from its checkpoint.

In [ ]:
from adelab.classroom import setup
from adelab import datagen
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

DA = setup("2.4", reset=True)
USERS, CLICKS, CDC = (f"{DA.paths.landing}/{d}" for d in ("users", "clicks", "cdc"))
datagen.land_users(USERS)
datagen.land_user_events(CLICKS)
spark.read.json(CLICKS).withColumn("event_ts", F.to_timestamp("event_ts")).write.saveAsTable("clicks_silver")

## Step 1 - Create the Bronze table with Change Data Feed enabled

In [ ]:
spark.sql("""
CREATE TABLE customers_bronze (
  user_id BIGINT, first_name STRING, last_name STRING, email STRING, city STRING, region STRING,
  updated_ts TIMESTAMP)
TBLPROPERTIES (delta.enableChangeDataFeed = true)""")

(spark.read.json(USERS)
      .select("user_id", "first_name", "last_name", "email", "city", "region",
              F.to_timestamp("signup_ts").alias("updated_ts"))
      .write.mode("append").saveAsTable("customers_bronze"))

In [ ]:
# CHECK step 1
props = {r.key: r.value for r in spark.sql("SHOW TBLPROPERTIES customers_bronze").collect()}
assert props.get("delta.enableChangeDataFeed") == "true", "Change Data Feed is not enabled"
assert spark.table("customers_bronze").count() == 200, "expected the 200 users of the initial load"
print("Step 1 passed")

## Step 2 - Apply a CDC batch to Bronze
Complete `apply_cdc_file()`: one `MERGE` that **deletes** rows where `operation = 'DELETE'` and **updates** `email`, `city`, `region`, `updated_ts` where `operation = 'UPDATE'`.

In [ ]:
def apply_cdc_file(path: str) -> None:
    cdc = spark.read.json(path).withColumn("updated_ts", F.to_timestamp("updated_ts"))
    (DeltaTable.forName(spark, "customers_bronze").alias("t")
        .merge(cdc.alias("s"), "t.user_id = s.user_id")
        .whenMatchedDelete(condition="s.operation = 'DELETE'")
        .whenMatchedUpdate(condition="s.operation = 'UPDATE'",
                           set={"email": "s.email", "city": "s.city", "region": "s.region",
                                "updated_ts": "s.updated_ts"})
        .execute())

apply_cdc_file(datagen.land_user_updates(CDC, batch=1))

In [ ]:
# CHECK step 2
changes = spark.sql("SELECT _change_type, count(*) AS n FROM table_changes('customers_bronze', 2) GROUP BY 1")
counts = {r._change_type: r.n for r in changes.collect()}
assert counts.get("delete", 0) > 0 and counts.get("update_postimage", 0) > 0,     f"the MERGE must both delete and update rows, change feed of version 2: {counts}"
assert counts.get("update_preimage") == counts.get("update_postimage"), "every update has a pre- and a post-image"
print("Step 2 passed", counts)

## Step 3 - Read the change feed
Return a DataFrame of the changes **from version 2 on** (batch read), without the pre-images.

In [ ]:
cdf = (spark.read.option("readChangeFeed", "true").option("startingVersion", 2)
       .table("customers_bronze").filter("_change_type != 'update_preimage'"))
display(cdf.select("_change_type", "_commit_version", "user_id", "email", "city"))

In [ ]:
# CHECK step 3
assert {"_change_type", "_commit_version", "_commit_timestamp"} <= set(cdf.columns), "read with readChangeFeed = true"
assert cdf.filter("_change_type = 'update_preimage'").count() == 0, "filter out the update_preimage rows"
assert cdf.agg(F.min("_commit_version")).first()[0] >= 2, "start at version 2 (startingVersion)"
print("Step 3 passed")

## Step 4 - Incremental propagation with a streaming CDF reader
Build `customers_silver` (email reduced to its domain, e.g. `example.com`), then complete `propagate()`:
- keep only the **latest** change per `user_id` in each micro-batch (highest `_commit_version`)
- `MERGE` into `customers_silver`: delete on `delete`, otherwise upsert
- delete the clicks of deleted users from `clicks_silver`

The stream must start reading **after** the initial load, because Silver gets the full snapshot first. Use `startingVersion` for that.

In [ ]:
spark.sql("CREATE TABLE customers_silver (user_id BIGINT, email_domain STRING, city STRING, region STRING, updated_ts TIMESTAMP)")
spark.sql("""INSERT INTO customers_silver
             SELECT user_id, substring_index(email, '@', -1), city, region, updated_ts
             FROM customers_bronze VERSION AS OF 1""")

def upsert_batch(batch_df, batch_id):
    latest = (batch_df.filter("_change_type != 'update_preimage'")
              .withColumn("rn", F.row_number().over(Window.partitionBy("user_id")
                                                    .orderBy(F.col("_commit_version").desc())))
              .filter("rn = 1")
              .select("user_id", "_change_type",
                      F.substring_index("email", "@", -1).alias("email_domain"),
                      "city", "region", "updated_ts"))
    (DeltaTable.forName(spark, "customers_silver").alias("t")
        .merge(latest.alias("s"), "t.user_id = s.user_id")
        .whenMatchedDelete(condition="s._change_type = 'delete'")
        .whenMatchedUpdateAll(condition="s._change_type != 'delete'")
        .whenNotMatchedInsertAll(condition="s._change_type != 'delete'")
        .execute())
    (DeltaTable.forName(spark, "clicks_silver").alias("c")
        .merge(latest.filter("_change_type = 'delete'").alias("d"), "c.user_id = d.user_id")
        .whenMatchedDelete()
        .execute())

def propagate():
    (spark.readStream.option("readChangeFeed", "true")
          .option("startingVersion", 2)
          .table("customers_bronze")
          .writeStream.foreachBatch(upsert_batch)
          .option("checkpointLocation", f"{DA.paths.checkpoints}/customers_cdf")
          .trigger(availableNow=True)
          .start().awaitTermination())

propagate()

In [ ]:
# CHECK step 4
bronze_ids = {r.user_id for r in spark.table("customers_bronze").select("user_id").collect()}
silver_ids = {r.user_id for r in spark.table("customers_silver").select("user_id").collect()}
assert bronze_ids == silver_ids, f"silver out of sync: {len(bronze_ids ^ silver_ids)} ids differ"
orphans = spark.table("clicks_silver").join(spark.table("customers_bronze"), "user_id", "left_anti").count()
assert orphans == 0, f"{orphans} clicks still belong to deleted users"
mismatch = spark.sql("""SELECT count(*) FROM customers_silver s JOIN customers_bronze b USING (user_id)
                        WHERE s.city <> b.city OR s.email_domain <> substring_index(b.email, '@', -1)""").first()[0]
assert mismatch == 0, f"{mismatch} rows not updated"
print("Step 4 passed")

## Step 5 - The next day
Apply the next CDC file and run **the same** `propagate()`. Only the new changes are processed.

In [ ]:
apply_cdc_file(datagen.land_user_updates(CDC, batch=2))
propagate()

In [ ]:
# CHECK step 5
bronze_ids = {r.user_id for r in spark.table("customers_bronze").select("user_id").collect()}
assert bronze_ids == {r.user_id for r in spark.table("customers_silver").select("user_id").collect()},     "customers_silver is out of sync after the second CDC file"
assert spark.table("clicks_silver").join(spark.table("customers_bronze"), "user_id", "left_anti").count() == 0,     "clicks of the newly deleted users are still in clicks_silver"
print("Step 5 passed")

## Step 6 - Make the deletes physical
Deleted customers can still be read with time travel (`VERSION AS OF 1`). Remove them for good from `customers_bronze` **and** `clicks_silver`. For this lab only, disable the retention safety check and vacuum with `RETAIN 0 HOURS`.

In [ ]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
for t in ("customers_bronze", "clicks_silver", "customers_silver"):
    spark.sql(f"REORG TABLE {t} APPLY (PURGE)")
    spark.sql(f"VACUUM {t} RETAIN 0 HOURS")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

In [ ]:
# CHECK step 6
# SELECT * (not count(*)): Delta answers count(*) from log statistics without reading any file
for table, version in (("customers_bronze", 1), ("clicks_silver", 0)):
    try:
        spark.sql(f"SELECT * FROM {table} VERSION AS OF {version}").collect()
    except Exception:
        continue
    raise AssertionError(f"{table} VERSION AS OF {version} is still readable - did you REORG ... PURGE and VACUUM it?")
print("Step 6 passed - historical versions are physically gone")

In [ ]:
DA.cleanup()